The following runs inside the BenchMAX environment.

# Slurm training and evaluation

BenchMAX supports deferring computations to *targets* that can be external to the physical machine running the interface code. An example is a **SlurmTarget**, which allows to use a cluster managed by the [Slurm](https://slurm.schedmd.com) workload manager.

## Setup

From the repository root:

```sh
cp -n .local/slurm-demo.example.json .local/slurm-demo.json
```

Edit `.local/slurm-demo.json` with your cluster's `host`, `user`, `control_path`, paths,
and Slurm account. Git ignores this file. BenchMAX must be installed in the configured
Python environment on the cluster.

Open an SSH connection in a terminal, enter your password, and leave it open:

```sh
ssh -o ControlMaster=yes -o ControlPath=<control_path> -o ControlPersist=10m <user>@<host>
```

Replace `<control_path>`, `<user>`, and `<host>` with the values from `.local/slurm-demo.json`.

In [1]:
import json
from pathlib import Path

# Load private settings from either the repository root or the notebooks directory.
repository = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
settings = json.loads((repository / ".local/slurm-demo.json").read_text())["target"]

HOST, USER_ACCOUNT, CONTROL_PATH = settings["host"], settings.get("user"), settings.get("control_path")
PYTHON, WORKSPACE, RUNS, ARTIFACTS = (settings[key] for key in ("python", "workspace", "runs", "artifacts"))
SLURM_ACCOUNT, PARTITION = settings["account"], settings["partition"]

In [2]:
from benchmax.execution import SlurmTarget

cluster = SlurmTarget(
    host=HOST, user=USER_ACCOUNT, control_path=CONTROL_PATH,
    python=PYTHON, workspace=WORKSPACE, runs=RUNS, artifacts=ARTIFACTS,

    account=SLURM_ACCOUNT, partition=PARTITION,
    cpus=1, memory_mb=2048, time_limit="00:02:00", gpus=0,
)
cluster.register("my_cluster", persist=True, replace=True);

The target is saved locally so the CLI can find `my_cluster`. Run this command with
`benchmax` from the notebook environment on your shell's `PATH`.
It checks MAPPO and its dependencies in the cluster's login environment.

In [3]:
!benchmax show mappo --target my_cluster

  Target: my_cluster
  Python     /shared/example_project/benchmax/.venv/bin/python
  workspace  /shared/example_project/benchmax
  scope      login environment; compute-node GPU/runtime availability is not tested

  mappo
  algorithm — based on benchmarl.mappo

  availability  available
  provider      benchmarl


## Train on the cluster

Train a tiny CPU test with RPS using eight frames.

In [4]:
from benchmax import load_game
from benchmax.training import train

game = load_game("rock_paper_scissors")
training_parameters = {
    "experiment": {
        "on_policy_collected_frames_per_batch": 8, "on_policy_n_envs_per_worker": 2,
        "on_policy_n_minibatch_iters": 1, "on_policy_minibatch_size": 4,
    },
    "model": {"num_cells": [8]},
}
training_job = train(
    game, "mappo", seed=7, device="cpu", total_frames=8, verbose=False, submit=True, params=training_parameters
)

In [ ]:
training_job.run("my_cluster");

## Monitor training

Inspect training without waiting for completion. This cell makes a read-only target request.

In [5]:
training_job.progress()

Training training-0000000000000001 — succeeded
  Slurm job: 100001
  Progress: 8 / 8 frames
Checked: 2000-01-01T00:00:00+00:00

### Optional cancellation

Optional cancellation test: **skip the next cell to continue the training → evaluation demo**. Running it cancels training and prevents dependent evaluation from running.

In [ ]:
training_job.cancel()

## Retrieve the policy reference

In [6]:
policy_reference = training_job.reference()
policy_reference

Job,training-0000000000000001
Target,Slurm · demo_user@demo-cluster
Output,policy
Run store,/scratch/demo_user/benchmax/runs


A `Reference[PolicyProfile]` can identify this job output or saved local files. Choose the operation explicitly:

```python
local_reference = policy_reference.save(path="./downloaded-policy")  # downloads files; no model loading
policy = local_reference.result()  # loads locally, without reopening the cluster job
# Or save and load together, always returning a PolicyProfile:
policy = policy_reference.result(path="./another-new-directory")
```

## Evaluate on the cluster

In [7]:
evaluation_job = game.evaluate(policy_reference, submit=True, target="my_cluster")

Inspect the evaluation and its policy dependency together. The tree shows whether training is queued, running, or finished, and whether evaluation is waiting for its policy or for resources. Percentages and ETA remain unavailable where no measurements are reported.

See the [progress guide](https://github.com/ALPIuzh/benchmax-preview/blob/main/src/benchmax/execution/README.md#progress) for programmatic snapshot access.

In [8]:
evaluation_job.progress()

Evaluation evaluation-0000000000000002 — running
  Slurm job: 100002
  Attempt elapsed: 62s
  Progress: not reported
  ETA: unavailable
└─ Training training-0000000000000001 — succeeded
     Slurm job: 100001
     Progress: 8 / 8 frames
Checked: 2000-01-01T00:00:01+00:00

In [9]:
metrics = evaluation_job.result()
metrics

{'expected_returns': {0: -0.0031867298282275414, 1: 0.0031867298282275414},
 'exploitability': 0.10401077568531036,
 'nash_conv': 0.20802155137062073,
 'nash_gap': 0.17937022177349426,
 'player_improvements': {0: 0.17937022177349426, 1: 0.028651329597126463}}